# Drawing & AOIs

`configure_draw(show=True)` puts a draw toolbar on the map — markers, lines,
rectangles, polygons and circles, with vertex editing, dragging and deletion
always available while it shows. Everything drawn lands in `m.drawings` as
GeoJSON features, the trait syncs **both ways**, and `m.draw_seq` bumps on every
create, edit and delete — the same one-observer pattern clicks use.

Needs `swiftmap`, `pandas`, and `ipywidgets` for the observer demos.

In [ ]:
import numpy as np
import pandas as pd
import ipywidgets as widgets
from swiftmap import Map

rng = np.random.default_rng(15)
n = 250
df = pd.DataFrame({
    "lat": 36.02 + rng.normal(0, 0.06, n),
    "lon": -5.45 + rng.normal(0, 0.10, n),
    "site": [f"Sensor {i:03d}" for i in range(n)],
    "reading": np.round(rng.gamma(4, 4, n), 1),
})

m = Map()
m.add_circle_markers(df, name="Sensors", color_col="reading")
m.configure_draw(show=True)
m

## Everything drawn comes back as GeoJSON

Draw a shape with the toolbar above, then watch the readout update. Circles
carry `properties.kind` and their radius, since GeoJSON has no circle; edits and
deletions bump `draw_seq` just like creations, so the handler never goes stale.

In [ ]:
draw_out = widgets.HTML("<i>Draw something with the toolbar above.</i>")

def on_draw(change):
    kinds = [d["properties"].get("kind", d["geometry"]["type"])
             for d in m.drawings]
    draw_out.value = (f"<b>{len(m.drawings)}</b> drawing(s): "
                      f"{', '.join(kinds) if kinds else '—'}")

m.observe(on_draw, names="draw_seq")
draw_out

## Seeding from Python

The trait writes in both directions: assign `m.drawings` and the shapes appear
on the map, editable like anything hand-drawn.

In [ ]:
patrol_box = {
    "type": "Feature",
    "properties": {"name": "Patrol box"},
    "geometry": {"type": "Polygon",
                 "coordinates": [[[-5.55, 35.98], [-5.35, 35.98],
                                  [-5.35, 36.08], [-5.55, 36.08],
                                  [-5.55, 35.98]]]},
}
m.drawings = [patrol_box]

## Using an AOI

An AOI is plain GeoJSON, so filtering against it is ordinary Python. A
rectangle needs only its bounding box; for arbitrary polygons, shapely's
`shape(aoi["geometry"]).contains(...)` is the one-liner.

In [ ]:
ring = np.asarray(patrol_box["geometry"]["coordinates"][0])
lon_min, lat_min = ring.min(axis=0)
lon_max, lat_max = ring.max(axis=0)

inside = df[df.lat.between(lat_min, lat_max) & df.lon.between(lon_min, lon_max)]
f"{len(inside)} of {len(df)} sensors inside the patrol box"

Point the map at the answer — spotlight the sensors inside (per-feature indices
line up with the DataFrame because this is one flat layer):

In [ ]:
m.set_feature_styles("Sensors", {int(i): {"color": "#ffcc00", "radius": 12}
                                 for i in inside.index});

## Clearing

Drag the box's vertices around and re-run the filter cell — edits land in
`m.drawings` like anything else. When the exercise is over:

In [ ]:
m.clear_drawings()
m.set_feature_styles("Sensors", {});

The live-app version of this loop — draw a box, watch a table filter itself —
is `shiny/03_draw_filter.py`. Exports (**08_export**) carry drawings and the
toolbar with them.